In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import random

torch.manual_seed(0)
np.random.seed(0)

In [ ]:
from PIL import Image

img = Image.open('knightley.png')
plt.imshow(img)
plt.axis('off')
img_array = np.array(img)
img_array = img_array / 256.0

In [ ]:
def normalize_image(i):
    i = torch.tensor(i)
    return (i - .5) * 2.

def denormalize_image(i):
    i = torch.tensor(i)
    return (i / 2.0) + 0.5

def show_image(img):
    x = img
    x = (x - x.min()) / (x.max() - x.min())
    plt.figure()
    plt.imshow(x)
    plt.axis("off")
    plt.show()

img_array_scaled = normalize_image(img_array)

In [ ]:
show_image(img_array_scaled)

In [ ]:
T = 25
beta_0 = 0
delta = 0.01

beta = [beta_0 + delta * t for t in range(T+1)]
beta = torch.tensor(beta)

alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha,0)

print(f"beta_T = {beta[T]:.4f}")
print(f"alpha_bar_T = {alpha_bar[T]:.4f}")
print(f"signal scale at T:  sqrt(alpha_bar_T) = {torch.sqrt(alpha_bar[T]):.4f}")
print(f"noise std at T:     sqrt(1 - alpha_bar_T) = {torch.sqrt(1 - alpha_bar[T]):.4f}")

In [ ]:
def forward_sample(x, t = None, show=False):
    t = random.randint(1, T) if t is None else t
    z = torch.randn_like(x)
    x_t = np.sqrt(alpha_bar[t])*x + np.sqrt(1-alpha_bar[t])*z
    if show:
        show_image(x_t)
    return x_t

In [ ]:
_ = forward_sample(img_array_scaled, 1, show=True)

In [ ]:
def reverse_sample(x_0, x_t, t, show=False):
    abar_t = alpha_bar[t]
    abar_tm1 = alpha_bar[t - 1]
    beta_t = beta[t]
    alpha_t = alpha[t]

    mu = (torch.sqrt(abar_tm1) * beta_t / (1 - abar_t)) * x_0 + (
        torch.sqrt(alpha_t) * (1 - abar_tm1) / (1 - abar_t)
    ) * x_t

    if t > 1:
        sigma = torch.sqrt((1 - abar_tm1) / (1 - abar_t) * beta_t)
    else:
        sigma = 0

    x = mu + sigma * torch.randn_like(x_t)
    if show:
        show_image(x_t)
    return x


t

In [ ]:
x_0 = img_array_scaled
x1 = forward_sample(x_0, 1, show=True)
x = reverse_sample(x_0, x1, 1, show=True)


In [ ]:
x_0 = img_array_scaled

x = x_0
for i in range(1,26):
    x = forward_sample(x, 1, show=True)

In [ ]:
for i in range(19,0,-1):
    x = reverse_sample(x_0, x, i, show=True)